## 子主题3

In [60]:
# 导入需要用到的库，主要用到的是networkx库用于生成有向图
import networkx as nx
import pandas as pd
import numpy as np
from tqdm import tqdm

In [61]:
# 读入数据
data_file_path = "../../Datasets/DB_LLM_processed.jsonl"
data = pd.read_json(data_file_path, lines=True)
data.head(3)

,dream_id,dream_summary,positive_or_not,entities_frequency,character_frequency,entity_sequence,metadata
0,izzy16_592,A dream about being at a video shop and encoun...,1.0,"{'video shop': 1, 'horror weekend': 1, 'greebl...","{'someone': 2, 'eugene': 1}","[video shop, someone, horror weekend, greeble ...","{'total_unique_entities': 5, 'total_unique_cha..."
1,izzy16_593,A dream about watching a show where characters...,1.0,"{'sliders': 1, 'portal': 1, 'room': 1, 'ball': 1}","{'quinn': 1, 'wade': 1}","[sliders, quinn, portal, wade, room, ball]","{'total_unique_entities': 4, 'total_unique_cha..."
2,izzy16_594,"A dream about being at school, discussing movi...",1.0,"{'school': 1, 'movie': 1, 'cop': 1, 'alejandro...","{'someone': 1, 'cop': 1, 'alejandro': 2}","[school, someone, movie, cop, alejandro, aleja...","{'total_unique_entities': 5, 'total_unique_cha..."


In [62]:
def gini(degree_seq):
    """
    计算基尼系数
    Input:
        degree_seq: 节点度的数组
    Return:
        gini_coeff: 基尼系数,取值范围在0~1的浮点数
        np.nan: 输入的数组为空, 或者求和为0
    """
    if len(degree_seq) == 0 or sum(degree_seq) == 0:
        return np.nan
    
    # 转为np数组
    degree_seq = np.array(degree_seq, dtype=np.float64)
    degree_seq = np.sort(degree_seq) 
    num = len(degree_seq)
    index_temp = np.arange(1, num+1)
    gini_coeff = np.sum((2*index_temp-num-1)*degree_seq) / (num*np.sum(degree_seq))
    return gini_coeff

In [63]:
def compute_coeff(data_row):
    """
    计算所有需要的系数，包装起来方便后续测试
    Input:
        data_row: 一行dataframe
    Return:
        cluster_coeff: 聚集系数，取值范围在0～1的浮点数
        avg_min_path: 平均最短路径
        gini_coeff: 基尼系数
        feedback_loops: 反馈环数量
    具体的定义可以见README.md
    """
    cluster_coeff = np.nan
    avg_min_path = np.nan
    gini_coeff = np.nan
    feedback_loops = 0

    seq = data_row["entity_sequence"]

    # 构建有向图
    G = nx.DiGraph()
    edges = [(seq[i], seq[i+1]) for i in range(len(seq)-1)]
    G.add_edges_from(edges)

    # 空图直接返回默认值
    if len(G) == 0:
        return cluster_coeff, avg_min_path, gini_coeff, feedback_loops

    G_undirected = G.to_undirected()

    # 计算聚集系数
    if len(G_undirected) == 1:
        cluster_coeff = 0.0
    else:
        degrees = [d for _, d in G_undirected.degree()]
        if max(degrees) < 2:
            cluster_coeff = 0.0
        else:
            cluster_coeff = nx.average_clustering(G_undirected)

    # 计算平均最短路径
    if len(G_undirected) > 1 and nx.is_connected(G_undirected):
        avg_min_path = nx.average_shortest_path_length(G_undirected)
    else:
        lengths = []
        for component in nx.connected_components(G_undirected):
            subgraph = G_undirected.subgraph(component)
            if len(subgraph) > 1:
                lengths.append(nx.average_shortest_path_length(subgraph))
        avg_min_path = max(lengths) if lengths else np.nan
    
    # 计算反馈环数量
    feedback_loops = len(list(nx.simple_cycles(G)))

    # 计算基尼系数
    degrees = [d for _, d in G.degree()]
    gini_coeff = gini(degrees)
    
    return cluster_coeff, avg_min_path, gini_coeff, feedback_loops

In [64]:
# 拿一行数据测试一下
row = data.iloc[1]
cluster_coeff, avg_min_path, gini_coeff, feedback_loops = compute_coeff(row)
print(f"cluster_coeff = {cluster_coeff}")
print(f"avg_min_path = {avg_min_path}")
print(f"gini = {gini_coeff}")
print(f"feedback_loops = {feedback_loops}")

cluster_coeff = 0.0
avg_min_path = 2.3333333333333335
gini = 0.13333333333333333
feedback_loops = 0


In [65]:
# 处理所有数据
saved_path = "../../Datasets/subtopic3/subtopic3.csv"
results = []
for idx, row in tqdm(data.iterrows(), total=len(data)):
    dream_id = row["dream_id"]
    positive_or_not = row["positive_or_not"]
    cluster_coeff, avg_min_path, gini_coeff, feedback_loops = compute_coeff(row)
    results.append({
        "dream_id": dream_id,
        "positive_or_not": positive_or_not,
        "cluster_coeff": cluster_coeff,
        "avg_min_path": avg_min_path,
        "feedback_loops": feedback_loops,
        "gini_coeff": gini_coeff
    })

df_results = pd.DataFrame(results)
df_results.to_csv(saved_path, index=False, encoding="utf-8")
print(f"已保存{len(df_results)}条数据")

100%|██████████| 27405/27405 [00:09<00:00, 2758.69it/s]

已保存27405条数据
